# Tutorial 03 · Regression and Classification with PyTorch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JimengShi/SysEng_Cornell/blob/main/Tutorial_03_PyTorch_Regression_and_Classification/tutorial_03_pytorch.ipynb)

**Run in your browser:** Click **Open in Colab**, connect, and select **Runtime → Run all**. The first code cell installs missing packages in Colab. No local installation is needed.

**Two questions, one training workflow:** Can vehicle measurements predict fuel economy? Can pixels identify a clothing category?

In this tutorial you will build a regression model for **Auto MPG** and a classifier for **Fashion-MNIST**. You will prepare data, define a neural network, train it, select a checkpoint using validation data, and interpret results on held-out test data.

**Prerequisites:** Python functions, lists, and basic arrays. No previous PyTorch experience is assumed.

**Runtime:** CPU is sufficient for the default quick run; a GPU is optional. Allow roughly 45–60 minutes to read and do the exercises. Execution time depends on hardware and download speed; these are planning estimates, not a Colab benchmark.

**Cost and access:** No API key, paid model, or Google Drive mount is required. You need internet access to download the public datasets (about 30 MB for Fashion-MNIST).

Adapted from the [PEESE SYSEN 5888/6888 tutorials](https://github.com/PEESEgroup/SysEn6888), with the upstream MIT license retained. This is an independently maintained teaching adaptation.


## Learning goals and route

By the end, you should be able to:

1. Explain the difference between regression and classification outputs and losses.
2. Use tensors, a `DataLoader`, `nn.Module`, and an optimizer together.
3. Split data before fitting preprocessing and keep the test set out of model selection.
4. Compare a learned model with a simple baseline and examine its mistakes.
5. Save model weights together with the preprocessing needed for inference.

**Route:** setup → tensors and gradients → fuel economy regression → clothing classification → save/reload → exercises.

### Start in Colab

1. Open the Colab link above and connect to a runtime. The default CPU runtime is enough.
2. Optionally save your own copy through **File → Save a copy in Drive**.
3. Select **Runtime → Run all**, or run cells in order with **Shift+Enter**.
4. For GPU use, change the runtime hardware accelerator to GPU and rerun from the top.

The notebook is self-contained: you do not need to clone this repository or run earlier tutorials. Colab runtimes are temporary; download any results you want to keep.


## 1. Prepare the environment

Colab normally provides the packages used here. This cell installs **only missing packages** and preserves an existing PyTorch/torchvision pair. We do not downgrade Colab's CUDA stack to an old pinned version. If an existing installation fails to import, start a fresh runtime before trying again.


In [ ]:
import importlib.util
import subprocess
import sys

packages = {"torch": "torch", "torchvision": "torchvision", "numpy": "numpy",
            "pandas": "pandas", "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if "torch" in missing or "torchvision" in missing:
    missing = sorted(set(missing + ["torch", "torchvision"]))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("All required packages are already installed.")


In [ ]:
import hashlib
import io
import json
import platform
import random
import time
import urllib.request
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision
from torchvision.datasets import FashionMNIST
from IPython.display import display

%matplotlib inline

SEED = 42
QUICK_RUN = True          # False uses more Fashion-MNIST images and training epochs.
DEVICE_PREFERENCE = "auto"  # "auto" uses CUDA if available; "cpu" always uses CPU.

assert DEVICE_PREFERENCE in {"auto", "cpu"}
device = torch.device("cuda" if DEVICE_PREFERENCE == "auto" and torch.cuda.is_available()
                      else "cpu")

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything()
# Small networks often run faster with a few CPU threads rather than many.
torch.set_num_threads(min(2, torch.get_num_threads()))
DATA_DIR = Path("data") / "tutorial_03"
OUTPUT_DIR = Path("outputs") / "tutorial_03"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})

versions = {"python": platform.python_version(), "torch": str(torch.__version__),
            "torchvision": str(torchvision.__version__), "numpy": np.__version__,
            "pandas": pd.__version__, "matplotlib": matplotlib.__version__,
            "device": str(device), "quick_run": QUICK_RUN, "seed": SEED}
display(pd.Series(versions, name="Environment"))
run_started = time.perf_counter()


Seeds help make comparisons reproducible, but floating-point results can still differ across library versions and devices. The environment table is part of your experimental record. **Change configuration values here, then rerun from the top.**

## 2. Tensors, layers, and automatic differentiation

A tensor is a multidimensional array. A linear layer transforms a batch with shape `[batch, features]` into `[batch, outputs]`. Its weights and biases are learned parameters.

PyTorch records operations on tensors that require gradients. Calling `backward()` computes derivatives; an optimizer then uses them to update the parameters.


In [ ]:
# Four observations, each with three features.
example_x = torch.randn(4, 3)
example_layer = nn.Linear(3, 1)
print("Input:", tuple(example_x.shape), "Output:", tuple(example_layer(example_x).shape))

# A tiny differentiable calculation: d(w - 3)^2 / dw = 2(w - 3).
w = torch.tensor(2.0, requires_grad=True)
toy_loss = (w - 3) ** 2
toy_loss.backward()
print(f"Loss = {toy_loss.item():.1f}; gradient at w=2 = {w.grad.item():.1f}")
assert w.grad.item() == -2.0


## 3. Regression: predict vehicle fuel economy

**Question:** Given vehicle characteristics, predict miles per gallon (MPG).

The [UCI Auto MPG dataset](https://archive.ics.uci.edu/dataset/9/auto+mpg) contains 398 historical vehicle records. Six records have missing horsepower; this introductory example drops them and reports the count. Dropping incomplete records is a teaching choice, not a universal missing-data strategy.

We use seven input variables: cylinders, displacement, horsepower, weight, acceleration, model year, and origin. `MPG` is the target. The vehicle name is excluded.

The download below checks a SHA-256 digest so that a changed file is not silently treated as the same dataset. Cached data are checked too.


In [ ]:
MPG_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"
MPG_SHA256 = "48b830e11feee5572525f8f1691ddb9d38d3d7b7063edcd8fca672c2a5e17d8d"
mpg_path = DATA_DIR / "auto-mpg.data"

def download_verified(url, destination, expected_sha256):
    if destination.exists():
        payload = destination.read_bytes()
    else:
        for attempt in range(3):
            try:
                with urllib.request.urlopen(url, timeout=60) as response:
                    payload = response.read()
                break
            except Exception as error:
                if attempt == 2:
                    raise RuntimeError(f"Download failed: {url}. Check connectivity and rerun.") from error
                time.sleep(attempt + 1)
    if hashlib.sha256(payload).hexdigest() != expected_sha256:
        raise RuntimeError(f"Checksum mismatch for {destination}. Remove the cached file and retry.")
    if not destination.exists():
        destination.write_bytes(payload)
    return payload

mpg_bytes = download_verified(MPG_URL, mpg_path, MPG_SHA256)
columns = ["MPG", "Cylinders", "Displacement", "Horsepower", "Weight",
           "Acceleration", "ModelYear", "Origin", "CarName"]
raw_mpg = pd.read_csv(io.BytesIO(mpg_bytes), sep=r"\s+", names=columns,
                      na_values="?", quotechar='"')
mpg = raw_mpg.drop(columns="CarName").dropna().reset_index(drop=True)
assert len(raw_mpg) == 398 and len(mpg) == 392
print(f"Loaded {len(raw_mpg)} records; removed {len(raw_mpg)-len(mpg)} incomplete records.")
display(mpg.head())


### 3.1 Split before fitting preprocessing

The **training set** is used to learn weights and preprocessing statistics. The **validation set** selects the checkpoint and model. The **test set** is reserved for the final comparison.

Here we use a seeded, approximately 70/15/15 random split. For a real forecasting problem, a time-based split may be more appropriate. This small historical dataset does not establish performance on modern vehicles.

Origin codes `1`, `2`, and `3` identify categories; they are not a numerical ranking. We encode them using three fixed indicator columns. One-hot encoding changes the representation; it does **not** itself make a linear model nonlinear.


In [ ]:
mpg_order = np.random.default_rng(SEED).permutation(len(mpg))
n_train = int(0.70 * len(mpg))
n_val = int(0.15 * len(mpg))
mpg_train_idx = mpg_order[:n_train]
mpg_val_idx = mpg_order[n_train:n_train+n_val]
mpg_test_idx = mpg_order[n_train+n_val:]
assert len(set(mpg_train_idx) & set(mpg_val_idx)) == 0
assert len(set(mpg_train_idx) & set(mpg_test_idx)) == 0
assert len(set(mpg_val_idx) & set(mpg_test_idx)) == 0
assert len(set(np.concatenate([mpg_train_idx, mpg_val_idx, mpg_test_idx]))) == len(mpg)

numeric_features = ["Cylinders", "Displacement", "Horsepower", "Weight",
                    "Acceleration", "ModelYear"]
feature_frame = mpg[numeric_features].copy()
for origin, name in [(1, "USA"), (2, "Europe"), (3, "Japan")]:
    feature_frame[f"Origin_{name}"] = (mpg["Origin"] == origin).astype("float32")
feature_names = feature_frame.columns.tolist()
X_mpg_raw = feature_frame.to_numpy(dtype=np.float32)
y_mpg_raw = mpg[["MPG"]].to_numpy(dtype=np.float32)
print(f"Train / validation / test: {len(mpg_train_idx)} / {len(mpg_val_idx)} / {len(mpg_test_idx)}")
print("Model inputs:", feature_names)

# Inspect only training data while designing the model.
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(mpg.loc[mpg_train_idx, "Weight"], mpg.loc[mpg_train_idx, "MPG"], alpha=0.65)
ax.set(xlabel="Vehicle weight (lb)", ylabel="Fuel economy (MPG)",
       title="Training data: weight and fuel economy")
plt.show()


### 3.2 Standardize using training statistics

For a feature $x$, standardization computes $z=(x-\mu_{train})/\sigma_{train}$. The same training mean and standard deviation are applied to validation and test data. Fitting these statistics on all data would leak information.

We also standardize the target to make optimization easier, then convert predictions back to MPG for reporting. A standard deviation of zero is replaced by one. Scaling helps optimization; it does not guarantee generalization or replace regularization.


In [ ]:
x_mean = X_mpg_raw[mpg_train_idx].mean(axis=0, keepdims=True)
x_std = X_mpg_raw[mpg_train_idx].std(axis=0, keepdims=True)
x_std = np.where(x_std < 1e-8, 1.0, x_std).astype(np.float32)
y_mean = y_mpg_raw[mpg_train_idx].mean(axis=0, keepdims=True)
y_std = y_mpg_raw[mpg_train_idx].std(axis=0, keepdims=True)
y_std = np.where(y_std < 1e-8, 1.0, y_std).astype(np.float32)

X_mpg = torch.from_numpy(((X_mpg_raw - x_mean) / x_std).astype(np.float32))
y_mpg = torch.from_numpy(((y_mpg_raw - y_mean) / y_std).astype(np.float32))
assert torch.isfinite(X_mpg).all() and torch.isfinite(y_mpg).all()
assert torch.allclose(X_mpg[mpg_train_idx].mean(dim=0), torch.zeros(X_mpg.shape[1]), atol=1e-5)

def make_loader(features, targets, batch_size=32, shuffle=False):
    return DataLoader(TensorDataset(features, targets), batch_size=batch_size,
                      shuffle=shuffle, num_workers=0,
                      generator=torch.Generator().manual_seed(SEED))

mpg_train_loader = make_loader(X_mpg[mpg_train_idx], y_mpg[mpg_train_idx], shuffle=True)
mpg_val_loader = make_loader(X_mpg[mpg_val_idx], y_mpg[mpg_val_idx])
print("Feature batch:", next(iter(mpg_train_loader))[0].shape)
print("Target batch:", next(iter(mpg_train_loader))[1].shape)


### 3.3 Define models and understand the training loop

We compare a **linear model** with a small **multilayer perceptron (MLP)**. `ReLU` introduces nonlinearity; the final regression layer has one output and no probability activation.

For every batch:

1. Clear old gradients with `zero_grad()`.
2. Compute predictions and a scalar loss.
3. Call `backward()` to calculate gradients.
4. Call `step()` to update weights.

An **epoch** visits all training examples once. A **batch** is one group used for an update. Validation uses `eval()` and disables gradient tracking; it never updates weights. The loss functions below use their default mean reduction, so epoch losses are weighted by the number of examples, including the last partial batch.


In [ ]:
class MPGRegressor(nn.Module):
    def __init__(self, n_features, hidden_size=None):
        super().__init__()
        self.network = (nn.Linear(n_features, 1) if hidden_size is None else
                        nn.Sequential(nn.Linear(n_features, hidden_size), nn.ReLU(),
                                      nn.Linear(hidden_size, hidden_size), nn.ReLU(),
                                      nn.Linear(hidden_size, 1)))

    def forward(self, x):
        return self.network(x)

def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss, count = 0.0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = loss_fn(outputs, targets)
        if not torch.isfinite(loss):
            raise ValueError("Non-finite training loss: inspect data and learning rate.")
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(inputs)
        count += len(inputs)
    return total_loss / count

@torch.no_grad()
def evaluate_loss(model, loader, loss_fn):
    model.eval()
    total_loss, count = 0.0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        loss = loss_fn(model(inputs), targets)
        if not torch.isfinite(loss):
            raise ValueError("Non-finite validation loss.")
        total_loss += loss.item() * len(inputs)
        count += len(inputs)
    return total_loss / count


We train for a fixed number of epochs and restore the checkpoint with the lowest validation loss. This lets us demonstrate overfitting without automatically deploying the final epoch. The helper below is shared by regression and classification, but each model has its own optimizer and loss.


In [ ]:
def fit(model, train_loader, val_loader, loss_fn, optimizer, epochs, print_every=25):
    history, best_loss, best_state, best_epoch = [], float("inf"), None, None
    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)
        val_loss = evaluate_loss(model, val_loader, loss_fn)
        history.append({"epoch": epoch, "train": train_loss, "validation": val_loss})
        if val_loss < best_loss:
            best_loss, best_epoch = val_loss, epoch
            best_state = {name: value.detach().cpu().clone()
                          for name, value in model.state_dict().items()}
        if epoch == 1 or epoch % print_every == 0 or epoch == epochs:
            print(f"Epoch {epoch:3d}: train={train_loss:.4f}, validation={val_loss:.4f}")
    model.load_state_dict(best_state)
    print(f"Restored epoch {best_epoch} (validation loss {best_loss:.4f}).")
    return pd.DataFrame(history), best_epoch

def plot_history(history, title, ylabel):
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(history["epoch"], history["train"], label="Training")
    ax.plot(history["epoch"], history["validation"], label="Validation")
    ax.set(xlabel="Epoch", ylabel=ylabel, title=title)
    ax.legend()
    plt.show()

regression_models, regression_histories, regression_best_epochs = {}, {}, {}
for name, hidden_size, learning_rate in [("Linear", None, 0.03), ("MLP", 32, 0.003)]:
    seed_everything()
    model = MPGRegressor(X_mpg.shape[1], hidden_size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    # Reset the shuffle generator for a comparable ordering across models.
    loader = make_loader(X_mpg[mpg_train_idx], y_mpg[mpg_train_idx], shuffle=True)
    print(f"\n{name} model")
    history, best_epoch = fit(model, loader, mpg_val_loader, nn.MSELoss(), optimizer,
                              epochs=150 if QUICK_RUN else 300)
    regression_models[name] = model
    regression_histories[name] = history
    regression_best_epochs[name] = best_epoch
    plot_history(history, f"{name}: fuel economy", "MSE (standardized target)")


### 3.4 Select on validation, then evaluate on test

The baseline always predicts the **training mean MPG**. It has no learned relationship with vehicle features.

We report **MAE** (mean absolute error) and **RMSE** (root mean squared error), both in MPG. RMSE penalizes large errors more strongly. Training minimizes standardized MSE; here we choose among the baseline and learned models using validation RMSE. We then show the prespecified test comparison without changing the selection.

Do not use the test results below to tune the learning rate, architecture, or number of epochs. Use the validation results for those decisions.


In [ ]:
@torch.no_grad()
def predict_mpg(model, features):
    model.eval()
    standardized = model(features.to(device)).cpu().numpy()
    return (standardized * y_std + y_mean).ravel()

def regression_metrics(actual, predicted):
    error = np.asarray(predicted) - np.asarray(actual)
    return {"MAE (MPG)": float(np.mean(np.abs(error))),
            "RMSE (MPG)": float(np.sqrt(np.mean(error ** 2)))}

val_predictions = {"Training-mean baseline": np.full(len(mpg_val_idx), float(y_mean.item()))}
val_predictions.update({name: predict_mpg(model, X_mpg[mpg_val_idx])
                        for name, model in regression_models.items()})
regression_validation = pd.DataFrame({
    name: regression_metrics(y_mpg_raw[mpg_val_idx].ravel(), pred)
    for name, pred in val_predictions.items()}).T
selected_regression = regression_validation["RMSE (MPG)"].idxmin()
print("Validation comparison:")
display(regression_validation.round(3))
print("Selected using validation RMSE:", selected_regression)

test_predictions = {"Training-mean baseline": np.full(len(mpg_test_idx), float(y_mean.item()))}
test_predictions.update({name: predict_mpg(model, X_mpg[mpg_test_idx])
                         for name, model in regression_models.items()})
regression_test = pd.DataFrame({
    name: regression_metrics(y_mpg_raw[mpg_test_idx].ravel(), pred)
    for name, pred in test_predictions.items()}).T
print("Held-out test comparison (not used for selection):")
display(regression_test.round(3))


In [ ]:
actual_mpg = y_mpg_raw[mpg_test_idx].ravel()
predicted_mpg = test_predictions[selected_regression]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(actual_mpg, predicted_mpg, alpha=0.75)
limits = [min(actual_mpg.min(), predicted_mpg.min()), max(actual_mpg.max(), predicted_mpg.max())]
axes[0].plot(limits, limits, "k--", label="Perfect prediction")
axes[0].set(xlabel="Observed MPG", ylabel="Predicted MPG", title=selected_regression)
axes[0].legend()
axes[1].scatter(predicted_mpg, predicted_mpg - actual_mpg, alpha=0.75)
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set(xlabel="Predicted MPG", ylabel="Prediction − observation (MPG)",
            title="Test residuals")
fig.tight_layout()
plt.show()


**Read the results:** Did either learned model beat the baseline? Do errors grow at high MPG? A smaller training loss alone does not prove better generalization. With only about 60 test vehicles, performance is sensitive to the split; these results are an introductory experiment, not a robust deployment estimate.

## 4. Classification: recognize clothing images

[Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist) has 60,000 training and 10,000 test grayscale images, each 28 × 28 pixels, in ten classes.

The default **quick run** uses 6,000 training images, 1,000 validation images, and 2,000 test images, balanced across classes. The **full run** uses 54,000 / 6,000 / 10,000. Validation images come only from the official training split. Quick-run test results are explicitly results on a subset, not on the full benchmark.

The dataset loader checks the published file checksums. We use an HTTPS mirror pinned to a commit in the original dataset repository. Both modes download the same compressed source files; the quick mode reduces computation, not download size.


In [ ]:
FASHION_REVISION = "b2617bb6d3ffa2e429640350f613e3291e10b141"

class HTTPSFashionMNIST(FashionMNIST):
    mirrors = [f"https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/{FASHION_REVISION}/data/fashion/"]

try:
    fashion_source_train = HTTPSFashionMNIST(root=str(DATA_DIR), train=True, download=True)
    fashion_source_test = HTTPSFashionMNIST(root=str(DATA_DIR), train=False, download=True)
except Exception as error:
    raise RuntimeError("Fashion-MNIST download/load failed. Check your connection and rerun this cell.") from error

class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

def stratified_indices(labels, per_class, seed):
    rng = np.random.default_rng(seed)
    groups = [[] for _ in per_class]
    for label in range(10):
        indices = rng.permutation(np.flatnonzero(np.asarray(labels) == label))
        assert sum(per_class) <= len(indices)
        start = 0
        for group, count in zip(groups, per_class):
            group.extend(indices[start:start + count].tolist())
            start += count
    return [rng.permutation(group) for group in groups]

fashion_train_idx, fashion_val_idx = stratified_indices(
    fashion_source_train.targets.numpy(), [600, 100] if QUICK_RUN else [5400, 600], SEED)
[fashion_test_idx] = stratified_indices(
    fashion_source_test.targets.numpy(), [200] if QUICK_RUN else [1000], SEED + 1)
assert not set(fashion_train_idx) & set(fashion_val_idx)

def fashion_tensors(source, indices):
    # Fixed pixel scaling; no statistics are fitted using validation/test images.
    images = source.data[indices].unsqueeze(1).float() / 255.0
    images = (images - 0.5) / 0.5  # [0, 255] -> [-1, 1]
    labels = source.targets[indices].long()
    return images, labels

fashion_x_train, fashion_y_train = fashion_tensors(fashion_source_train, fashion_train_idx)
fashion_x_val, fashion_y_val = fashion_tensors(fashion_source_train, fashion_val_idx)
fashion_x_test, fashion_y_test = fashion_tensors(fashion_source_test, fashion_test_idx)
fashion_train_loader = make_loader(fashion_x_train, fashion_y_train, batch_size=128, shuffle=True)
fashion_val_loader = make_loader(fashion_x_val, fashion_y_val, batch_size=256)
fashion_test_loader = make_loader(fashion_x_test, fashion_y_test, batch_size=256)
print("Train / validation / test:", len(fashion_y_train), len(fashion_y_val), len(fashion_y_test))
print("Image tensor:", fashion_x_train.shape, "Label dtype:", fashion_y_train.dtype)


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for label, ax in enumerate(axes.flat):
    i = int((fashion_y_train == label).nonzero()[0].item())
    ax.imshow((fashion_x_train[i, 0] + 1) / 2, cmap="gray", vmin=0, vmax=1)
    ax.set_title(class_names[label], fontsize=9)
    ax.axis("off")
fig.suptitle("One training image from each class")
fig.tight_layout()
plt.show()


### 4.1 From pixels to logits

`Flatten` converts each `[1, 28, 28]` image to 784 inputs. A hidden layer learns 128 features, and the last layer returns ten **logits** (unrestricted class scores).

`CrossEntropyLoss` expects logits of shape `[batch, 10]` and integer class labels of shape `[batch]`. **Do not apply softmax before this loss**; it already performs the required log-softmax calculation. Apply softmax only when you want to display class probabilities after training.

| Task | Output per example | Target dtype | Training loss | Reported metric |
|---|---|---|---|---|
| Regression | One number | `float32` | MSE | MAE / RMSE in MPG |
| Classification | Ten logits | `int64` (`long`) | Cross-entropy | Accuracy and confusion matrix |


In [ ]:
class FashionClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, 128),
                                     nn.ReLU(), nn.Linear(128, 10))

    def forward(self, x):
        return self.network(x)

seed_everything()
fashion_model = FashionClassifier().to(device)
fashion_optimizer = torch.optim.Adam(fashion_model.parameters(), lr=0.001)
fashion_loss = nn.CrossEntropyLoss()
print(fashion_model)
assert fashion_model(fashion_x_train[:4].to(device)).shape == (4, 10)

fashion_history, fashion_best_epoch = fit(
    fashion_model, fashion_train_loader, fashion_val_loader, fashion_loss,
    fashion_optimizer, epochs=5 if QUICK_RUN else 10, print_every=1)
plot_history(fashion_history, "Fashion-MNIST classifier", "Cross-entropy")


### 4.2 Evaluate accuracy and inspect confusion

Accuracy is the proportion of correct predictions. A majority-class baseline always returns the most frequent training label; because these subsets are balanced, it achieves 10% here. A confusion matrix shows which classes the model confuses, which a single accuracy value hides.


In [ ]:
@torch.no_grad()
def collect_logits(model, loader):
    model.eval()
    logits, labels = [], []
    for images, targets in loader:
        logits.append(model(images.to(device)).cpu())
        labels.append(targets.cpu())
    return torch.cat(logits), torch.cat(labels)

fashion_val_logits, fashion_val_targets = collect_logits(fashion_model, fashion_val_loader)
fashion_test_logits, fashion_test_targets = collect_logits(fashion_model, fashion_test_loader)
fashion_test_pred = fashion_test_logits.argmax(dim=1)
fashion_val_accuracy = (fashion_val_logits.argmax(dim=1) == fashion_val_targets).float().mean().item()
fashion_test_accuracy = (fashion_test_pred == fashion_test_targets).float().mean().item()
majority_class = torch.bincount(fashion_y_train).argmax().item()
baseline_accuracy = (fashion_test_targets == majority_class).float().mean().item()
print(f"Validation accuracy: {fashion_val_accuracy:.1%}")
print(f"Test accuracy ({len(fashion_test_targets):,} images): {fashion_test_accuracy:.1%}")
print(f"Majority-class baseline: {baseline_accuracy:.1%}")

confusion = torch.bincount(10 * fashion_test_targets + fashion_test_pred,
                           minlength=100).reshape(10, 10).numpy()
assert confusion.sum() == len(fashion_test_targets)
fig, ax = plt.subplots(figsize=(9, 7))
image = ax.imshow(confusion, cmap="Blues")
ax.set(xticks=range(10), yticks=range(10), xticklabels=class_names,
       yticklabels=class_names, xlabel="Predicted class", ylabel="True class",
       title=f"Fashion-MNIST: {len(fashion_test_targets):,} held-out test images")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
for row in range(10):
    for col in range(10):
        ax.text(col, row, str(confusion[row, col]), ha="center", va="center", fontsize=8,
                color="white" if confusion[row, col] > confusion.max() / 2 else "black")
fig.colorbar(image, ax=ax, label="Image count")
fig.tight_layout()
plt.show()


### 4.3 Look at mistakes, not only successful predictions

The next plot shows the first eight errors in the fixed test ordering, rather than hand-picked examples. Softmax probabilities are model scores, **not a guarantee of correctness or calibrated confidence**. A confident mistake is still a mistake.


In [ ]:
fashion_probabilities = fashion_test_logits.softmax(dim=1)
assert torch.allclose(fashion_probabilities.sum(dim=1), torch.ones(len(fashion_test_targets)), atol=1e-5)
mistakes = (fashion_test_pred != fashion_test_targets).nonzero().flatten()[:8]
if len(mistakes) == 0:
    print("No errors in this test subset; inspect a larger held-out set before drawing conclusions.")
else:
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for ax in axes.flat:
        ax.axis("off")
    for ax, index in zip(axes.flat, mistakes.tolist()):
        pred, actual = fashion_test_pred[index].item(), fashion_test_targets[index].item()
        ax.imshow((fashion_x_test[index, 0] + 1) / 2, cmap="gray", vmin=0, vmax=1)
        score = fashion_probabilities[index, pred].item()
        ax.set_title(f"True: {class_names[actual]}\nPred: {class_names[pred]} ({score:.0%})", fontsize=9)
    fig.suptitle("First test errors: probabilities can be confidently wrong")
    fig.tight_layout()
    plt.show()


## 5. Save and reload a model

Weights are only part of a prediction pipeline. A regression checkpoint also needs the exact feature order, category encoding, and training-derived normalization. We save those with the validation-selected neural regressor; if the mean baseline wins the overall comparison, that fact remains in the results.

The reload check uses CPU even if training used a GPU. `weights_only=True` loads the tensor-and-basic-metadata checkpoints written here. Only load checkpoints from sources you trust.


In [ ]:
selected_neural_regression = regression_validation.loc[list(regression_models), "RMSE (MPG)"].idxmin()
mpg_checkpoint = {
    "model_state": {k: v.detach().cpu() for k, v in regression_models[selected_neural_regression].state_dict().items()},
    "hidden_size": None if selected_neural_regression == "Linear" else 32,
    "feature_names": feature_names,
    "origin_encoding": {"1": "Origin_USA", "2": "Origin_Europe", "3": "Origin_Japan"},
    "x_mean": torch.from_numpy(x_mean), "x_std": torch.from_numpy(x_std),
    "y_mean": torch.from_numpy(y_mean), "y_std": torch.from_numpy(y_std),
    "seed": SEED,
}
torch.save(mpg_checkpoint, OUTPUT_DIR / "mpg_regressor.pt")
torch.save({"model_state": {k: v.detach().cpu() for k, v in fashion_model.state_dict().items()},
            "class_names": class_names, "pixel_scaling": "(pixel / 255.0 - 0.5) / 0.5",
            "seed": SEED}, OUTPUT_DIR / "fashion_classifier.pt")

loaded_mpg = torch.load(OUTPUT_DIR / "mpg_regressor.pt", map_location="cpu", weights_only=True)
restored_mpg = MPGRegressor(len(loaded_mpg["feature_names"]), loaded_mpg["hidden_size"])
restored_mpg.load_state_dict(loaded_mpg["model_state"])
restored_mpg.eval()
raw_examples = torch.from_numpy(X_mpg_raw[mpg_test_idx[:5]])
with torch.no_grad():
    scaled_examples = (raw_examples - loaded_mpg["x_mean"]) / loaded_mpg["x_std"]
    restored_predictions = (restored_mpg(scaled_examples) * loaded_mpg["y_std"] + loaded_mpg["y_mean"])
np.testing.assert_allclose(restored_predictions.numpy().ravel(),
                           test_predictions[selected_neural_regression][:5], rtol=1e-4, atol=1e-4)

loaded_fashion = torch.load(OUTPUT_DIR / "fashion_classifier.pt", map_location="cpu", weights_only=True)
restored_fashion = FashionClassifier()
restored_fashion.load_state_dict(loaded_fashion["model_state"])
restored_fashion.eval()
with torch.no_grad():
    restored_logits = restored_fashion(fashion_x_test[:5])
torch.testing.assert_close(restored_logits, fashion_test_logits[:5], rtol=1e-4, atol=1e-4)
print("Both saved models reproduce their original predictions within numerical tolerance.")


In [ ]:
results = {
    "environment": versions,
    "data": {"auto_mpg_sha256": MPG_SHA256, "fashion_mnist_revision": FASHION_REVISION},
    "regression": {"split_sizes": [len(mpg_train_idx), len(mpg_val_idx), len(mpg_test_idx)],
                   "validation": regression_validation.to_dict(orient="index"),
                   "test": regression_test.to_dict(orient="index"),
                   "selected_by_validation": selected_regression,
                   "best_epochs": regression_best_epochs},
    "classification": {"split_sizes": [len(fashion_y_train), len(fashion_y_val), len(fashion_y_test)],
                       "validation_accuracy": fashion_val_accuracy,
                       "test_accuracy": fashion_test_accuracy,
                       "baseline_accuracy": baseline_accuracy, "best_epoch": fashion_best_epoch},
    "elapsed_seconds_excluding_setup": round(time.perf_counter() - run_started, 2),
}
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(results, indent=2, allow_nan=False))
print(f"Run complete in {results['elapsed_seconds_excluding_setup']:.1f} seconds after setup.")
print("Saved files:")
for artifact in sorted(OUTPUT_DIR.iterdir()):
    print(" -", artifact)


In Colab, open the **Files** sidebar and download files from `outputs/tutorial_03/` before the runtime is discarded. The saved notebook and runtime files are separate; saving a notebook does not automatically preserve the trained weights.

## 6. Exercises

Keep a small experiment log: hypothesis, one change, validation result, and interpretation. Avoid repeatedly inspecting the test set while tuning.

1. **Explain the shapes.** Why is the regression target `[batch, 1]` but the classification target `[batch]`? What happens if the regression target accidentally broadcasts across a batch?
2. **Change the learning rate.** Try `0.0003` and `0.01` for the regression MLP. Compare validation curves, keeping the split fixed. Does faster initial learning give the best validation result?
3. **Change capacity.** Replace the regression hidden width of 32 with 8 or 64. Also update the saved architecture metadata if you change it. Look for a growing gap between training and validation loss.
4. **Study a scheduler (extension).** Try `torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)` for regression. Step it once per epoch after the optimizer updates, and compare with the fixed-rate run.
5. **Study classification errors.** Which class pair is most frequently confused? Propose a reason from the images. Distinguish evidence from speculation.
6. **Run the larger experiment (extension).** Set `QUICK_RUN=False` and rerun from the top. Compare validation performance and runtime. The test subset also changes, so the quick/full test scores are not a controlled measure of the effect of training-set size alone.

<details>
<summary>Concept checks: reveal after answering</summary>

- Fit normalization on training data only; reuse those statistics unchanged.
- `model.eval()` changes evaluation behavior for layers such as dropout; it does not disable gradients by itself. Use `no_grad()` or `inference_mode()` too.
- `zero_grad()` prevents unintended accumulation from earlier batches.
- Regression predicts a continuous quantity; a classifier returns one score per class.
- For class-index targets, cross-entropy takes integer labels and raw logits.
- Softmax scores need calibration assessment before being interpreted as reliable confidence.
- A more complex model is useful only if evaluation supports the added complexity.

</details>

## Troubleshooting

| Symptom | What to do |
|---|---|
| Import or package compatibility error | Start a fresh Colab runtime and run from the first cell; avoid installing unrelated packages into the session. |
| Dataset download fails | Check internet access and rerun the download cell. The error identifies which dataset failed. |
| No GPU available | Keep the default quick run; CPU is supported. |
| `NameError` after reconnecting | Runtime state was lost. Run all cells from the top. |
| GPU memory error after experiments | Reconnect to a fresh runtime, keep `QUICK_RUN=True`, or set `DEVICE_PREFERENCE="cpu"`. |
| Metrics differ slightly | Record the printed versions and device; exact results may vary across environments. |

## Sources and attribution

- Original teaching material: [PEESEgroup/SysEn6888, Tutorial 03](https://github.com/PEESEgroup/SysEn6888/tree/0b326ded95e415d7a60660879aaa737e3c7d79ea/Tutorial_03_PyTorch_and_TensorFlow_Introduction). Copyright (c) 2025 PEESE, MIT license; see the repository `LICENSE`.
- Auto MPG: R. Quinlan (1993), [UCI Machine Learning Repository](https://doi.org/10.24432/C5859H), CC BY 4.0. Data are downloaded from UCI, not included in this notebook.
- Fashion-MNIST: Han Xiao, Kashif Rasul, Roland Vollgraf (2017), [dataset and license](https://github.com/zalandoresearch/fashion-mnist), MIT license. Data are downloaded from the original dataset repository.
- [PyTorch basics](https://docs.pytorch.org/tutorials/beginner/basics/intro.html), [CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html), and [saving and loading models](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html).
- [Colab FAQ](https://research.google.com/colaboratory/faq.html): runtime behavior, sharing, and resource availability.
